In [ ]:
# -*- coding: utf-8 -*-

import os
import json
import glob
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from shapely.geometry import Polygon


TARGET_CRS = "EPSG:2326"

HK_FULL_BOUNDS = {
    "xmin": 800000,
    "xmax": 865000,
    "ymin": 800000,
    "ymax": 848000,
}

KOWLOON_BOUNDS = {
    "xmin": 834000,
    "xmax": 841500,
    "ymin": 817000,
    "ymax": 823500,
}

MACRO_BASEMAP = cx.providers.CartoDB.Voyager
MICRO_BASEMAP = cx.providers.Esri.WorldImagery

COLORS = {
    "LandsD": "#485DAF",
    "BDBIAR": "#B92A4A",
    "OSM": "#E67E22",
    "Overture": "#81739C",
}

STYLES = {
    "LandsD_Full": {
        "color": COLORS["LandsD"],
        "edgecolor": "none",
        "alpha": 0.95,
    },
    "LandsD_Zoom": {
        "color": COLORS["LandsD"],
        "edgecolor": "white",
        "linewidth": 0.6,
        "alpha": 0.8,
    },
    "BDBIAR_Full": {
        "color": COLORS["BDBIAR"],
        "markersize": 1.5,
        "edgecolor": "none",
        "alpha": 0.6,
    },
    "BDBIAR_Zoom": {
        "color": COLORS["BDBIAR"],
        "markersize": 15,
        "edgecolor": "none",
        "alpha": 0.8,
    },
    "OSM_Full": {
        "color": COLORS["OSM"],
        "markersize": 1.5,
        "edgecolor": "none",
        "alpha": 0.5,
    },
    "OSM_Zoom": {
        "color": COLORS["OSM"],
        "markersize": 15,
        "edgecolor": "none",
        "alpha": 0.7,
    },
    "Overture_Full": {
        "color": COLORS["Overture"],
        "markersize": 1.5,
        "edgecolor": "none",
        "alpha": 0.5,
    },
    "Overture_Zoom": {
        "color": COLORS["Overture"],
        "markersize": 15,
        "edgecolor": "none",
        "alpha": 0.7,
    },
}


def resolve_project_root() -> Path:
    """
    Resolve the project root when config.py cannot be imported.
    """
    start = Path.cwd()
    candidates = [start, start.parent]

    for candidate in candidates:
        if (candidate / "output").exists() and (candidate / "intermediate_files").exists():
            return candidate

    return start


def first_existing_path(candidates):
    """
    Return the first existing path from a list of candidates.
    """
    for path in candidates:
        if path is not None and Path(path).exists():
            return Path(path)
    return None


def load_project_paths():
    """
    Load paths from config.py when available.
    Fallback paths are provided for notebook/script portability.
    """
    try:
        from config import (
            BASE_DIR as CONFIG_BASE_DIR,
            OFFICIAL_LIB_BASE_PATH,
            BDBIAR_CACHE_PATH,
            OSM_ALL_PATH,
            OVERTURE_PARQUET_FILE_PATH,
            OVERTURE_CLEAN_PATH,
            OZP_DATA_DIR,
        )

        base_dir = Path(CONFIG_BASE_DIR)

        paths = {
            "BASE_DIR": base_dir,
            "PLOT_DIR": base_dir / "plot",
            "LandsD": first_existing_path([OFFICIAL_LIB_BASE_PATH]),
            "BDBIAR": first_existing_path([BDBIAR_CACHE_PATH]),
            "OSM": first_existing_path([OSM_ALL_PATH]),
            "Overture_Clean": first_existing_path([OVERTURE_CLEAN_PATH]),
            "Overture_Parquet": first_existing_path([OVERTURE_PARQUET_FILE_PATH]),
            "OZP_DIR": first_existing_path([OZP_DATA_DIR]),
        }
        return paths

    except ImportError:
        base_dir = resolve_project_root()
        intermediate_dir = base_dir / "intermediate_files"
        data_dir = base_dir / "data"

        paths = {
            "BASE_DIR": base_dir,
            "PLOT_DIR": base_dir / "plot",
            "LandsD": first_existing_path([
                intermediate_dir / "step1_official_building_library_base.geojson",
                intermediate_dir / "gdf_official_library_base.geojson",
            ]),
            "BDBIAR": first_existing_path([
                intermediate_dir / "step1_bdbiar_points.geojson",
                intermediate_dir / "bdbiar.geojson",
            ]),
            "OSM": first_existing_path([
                intermediate_dir / "step1_osm_features_all.geojson",
                intermediate_dir / "osm_all.geojson",
            ]),
            "Overture_Clean": first_existing_path([
                intermediate_dir / "step1_overture_places_clean.geojson",
            ]),
            "Overture_Parquet": first_existing_path([
                data_dir / "overture" / "places.parquet",
            ]),
            "OZP_DIR": first_existing_path([
                data_dir / "OZP_Zones",
            ]),
        }
        return paths


def set_plot_style():
    """
    Apply a clean publication-oriented plotting style.
    """
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
        "font.size": 12,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.linewidth": 1.0,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    })


def ensure_target_crs(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Ensure a GeoDataFrame uses the target CRS.
    """
    if gdf is None or gdf.empty:
        return gdf

    if gdf.crs is None:
        gdf = gdf.set_crs(TARGET_CRS)

    if str(gdf.crs) != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)

    return gdf


def convert_polygons_to_centroids_if_needed(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Convert polygon-like geometries to centroids for point-based display layers.
    """
    if gdf is None or gdf.empty:
        return gdf

    geom_types = set(gdf.geometry.geom_type.dropna().unique())
    polygon_like = {"Polygon", "MultiPolygon"}

    if geom_types.issubset(polygon_like):
        gdf = gdf.copy()
        gdf["geometry"] = gdf.geometry.centroid

    return gdf


def load_vector_file(path: Path) -> gpd.GeoDataFrame:
    """
    Load a vector dataset and convert it to the target CRS.
    """
    if path is None or not path.exists():
        return gpd.GeoDataFrame()

    gdf = gpd.read_file(path)
    return ensure_target_crs(gdf)


def load_overture_data(clean_geojson_path: Path = None, parquet_path: Path = None) -> gpd.GeoDataFrame:
    """
    Load Overture data, preferring the cleaned GeoJSON if available.
    """
    if clean_geojson_path is not None and clean_geojson_path.exists():
        return load_vector_file(clean_geojson_path)

    if parquet_path is None or not parquet_path.exists():
        return gpd.GeoDataFrame()

    print("[INFO] Loading Overture parquet...")
    try:
        gdf = gpd.read_parquet(parquet_path)
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326")
        return gdf.to_crs(TARGET_CRS)

    except Exception:
        df = pd.read_parquet(parquet_path)
        try:
            from shapely import wkb
        except ImportError:
            return gpd.GeoDataFrame()

        if "geometry" not in df.columns:
            return gpd.GeoDataFrame()

        df["geometry"] = df["geometry"].apply(lambda x: wkb.loads(x) if x else None)
        gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
        return gdf.to_crs(TARGET_CRS)


def load_ozp_esri_jsons(folder_path: Path) -> gpd.GeoDataFrame:
    """
    Load OZP Esri JSON files from a folder and convert them into a GeoDataFrame.
    """
    if folder_path is None or not folder_path.exists():
        return gpd.GeoDataFrame()

    print("[INFO] Loading OZP Esri JSON files...")
    features = []
    json_files = glob.glob(os.path.join(str(folder_path), "*.json"))

    if not json_files:
        print(f"[WARN] No OZP JSON files found in {folder_path}")
        return gpd.GeoDataFrame()

    for json_file in json_files:
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        for feature in data.get("features", []):
            attributes = feature.get("attributes", {})
            geometry = feature.get("geometry", {})

            if "rings" not in geometry or not geometry["rings"]:
                continue

            try:
                polygon = Polygon(geometry["rings"][0])
                features.append({
                    "geometry": polygon,
                    "DESC_ENG": attributes.get("DESC_ENG", "Unknown"),
                })
            except Exception:
                continue

    if not features:
        return gpd.GeoDataFrame()

    gdf = gpd.GeoDataFrame(features, crs=TARGET_CRS)
    return ensure_target_crs(gdf)


def plot_single_map(
    gdf: gpd.GeoDataFrame,
    title: str,
    filename: Path,
    bounds: dict,
    style_kwargs: dict,
    color_col: str = None,
    zoom_level: int = 12,
    basemap_source=None,
):
    """
    Plot a single map and save it to disk.
    """
    if gdf is None or gdf.empty:
        print(f"[WARN] Skipping empty layer: {title}")
        return

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.set_xlim(bounds["xmin"], bounds["xmax"])
    ax.set_ylim(bounds["ymin"], bounds["ymax"])

    plot_kwargs = style_kwargs.copy()

    if color_col and color_col in gdf.columns:
        gdf.plot(ax=ax, column=color_col, legend=False, **plot_kwargs)
    else:
        gdf.plot(ax=ax, **plot_kwargs)

    try:
        if basemap_source is None:
            basemap_source = cx.providers.CartoDB.Voyager

        cx.add_basemap(
            ax,
            crs=TARGET_CRS,
            source=basemap_source,
            zoom=zoom_level,
            alpha=0.8,
            attribution="",
        )
    except Exception as e:
        print(f"[WARN] Basemap loading failed for {title}: {e}")

    ax.set_title(
        title,
        fontsize=18,
        pad=15,
        fontweight="bold",
    )
    ax.axis("off")

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    print(f"[INFO] Saved: {filename}")


def run_all_maps():
    """
    Generate the full set of 10 source-overview maps.
    """
    paths = load_project_paths()
    plot_dir = paths["PLOT_DIR"]
    plot_dir.mkdir(parents=True, exist_ok=True)

    print("[INFO] Project base directory:", paths["BASE_DIR"])
    print("[INFO] Plot output directory:", plot_dir)

    set_plot_style()

    # 1. LandsD
    if paths["LandsD"] is not None and paths["LandsD"].exists():
        print("[1/5] Processing LandsD...")
        gdf_landsd = load_vector_file(paths["LandsD"])

        plot_single_map(
            gdf_landsd,
            "LandsD Building Footprints (Full HK)",
            plot_dir / "Fig2a_LandsD_Full.png",
            HK_FULL_BOUNDS,
            STYLES["LandsD_Full"],
            zoom_level=12,
            basemap_source=MACRO_BASEMAP,
        )
        plot_single_map(
            gdf_landsd,
            "LandsD Building Footprints (Kowloon)",
            plot_dir / "Fig2a_LandsD_Zoom.png",
            KOWLOON_BOUNDS,
            STYLES["LandsD_Zoom"],
            zoom_level=15,
            basemap_source=MICRO_BASEMAP,
        )
    else:
        print("[WARN] LandsD source not found.")

    # 2. BDBIAR
    if paths["BDBIAR"] is not None and paths["BDBIAR"].exists():
        print("[2/5] Processing BDBIAR...")
        gdf_bdbiar = load_vector_file(paths["BDBIAR"])
        gdf_bdbiar = convert_polygons_to_centroids_if_needed(gdf_bdbiar)

        plot_single_map(
            gdf_bdbiar,
            "BDBIAR Semantic Records (Full HK)",
            plot_dir / "Fig2a_BDBIAR_Full.png",
            HK_FULL_BOUNDS,
            STYLES["BDBIAR_Full"],
            zoom_level=12,
            basemap_source=MACRO_BASEMAP,
        )
        plot_single_map(
            gdf_bdbiar,
            "BDBIAR Semantic Records (Kowloon)",
            plot_dir / "Fig2a_BDBIAR_Zoom.png",
            KOWLOON_BOUNDS,
            STYLES["BDBIAR_Zoom"],
            zoom_level=15,
            basemap_source=MICRO_BASEMAP,
        )
    else:
        print("[WARN] BDBIAR source not found.")

    # 3. OSM
    if paths["OSM"] is not None and paths["OSM"].exists():
        print("[3/5] Processing OSM...")
        gdf_osm = load_vector_file(paths["OSM"])
        gdf_osm = convert_polygons_to_centroids_if_needed(gdf_osm)

        plot_single_map(
            gdf_osm,
            "OpenStreetMap Coverage (Full HK)",
            plot_dir / "Fig2a_OSM_Full.png",
            HK_FULL_BOUNDS,
            STYLES["OSM_Full"],
            zoom_level=12,
            basemap_source=MACRO_BASEMAP,
        )
        plot_single_map(
            gdf_osm,
            "OpenStreetMap Coverage (Kowloon)",
            plot_dir / "Fig2a_OSM_Zoom.png",
            KOWLOON_BOUNDS,
            STYLES["OSM_Zoom"],
            zoom_level=15,
            basemap_source=MICRO_BASEMAP,
        )
    else:
        print("[WARN] OSM source not found.")

    # 4. Overture
    overture_available = (
        (paths["Overture_Clean"] is not None and paths["Overture_Clean"].exists())
        or (paths["Overture_Parquet"] is not None and paths["Overture_Parquet"].exists())
    )

    if overture_available:
        print("[4/5] Processing Overture...")
        gdf_overture = load_overture_data(
            clean_geojson_path=paths["Overture_Clean"],
            parquet_path=paths["Overture_Parquet"],
        )
        gdf_overture = convert_polygons_to_centroids_if_needed(gdf_overture)

        plot_single_map(
            gdf_overture,
            "Overture Maps Places (Full HK)",
            plot_dir / "Fig2a_Overture_Full.png",
            HK_FULL_BOUNDS,
            STYLES["Overture_Full"],
            zoom_level=12,
            basemap_source=MACRO_BASEMAP,
        )
        plot_single_map(
            gdf_overture,
            "Overture Maps Places (Kowloon)",
            plot_dir / "Fig2a_Overture_Zoom.png",
            KOWLOON_BOUNDS,
            STYLES["Overture_Zoom"],
            zoom_level=15,
            basemap_source=MICRO_BASEMAP,
        )
    else:
        print("[WARN] Overture source not found.")

    # 5. OZP
    if paths["OZP_DIR"] is not None and paths["OZP_DIR"].exists():
        print("[5/5] Processing OZP...")
        gdf_ozp = load_ozp_esri_jsons(paths["OZP_DIR"])

        style_ozp_full = {
            "edgecolor": "none",
            "alpha": 0.65,
            "cmap": "tab20",
        }
        style_ozp_zoom = {
            "edgecolor": "white",
            "linewidth": 0.5,
            "alpha": 0.45,
            "cmap": "tab20",
        }

        plot_single_map(
            gdf_ozp,
            "Outline Zoning Plan (Full HK)",
            plot_dir / "Fig2a_OZP_Full.png",
            HK_FULL_BOUNDS,
            style_ozp_full,
            color_col="DESC_ENG",
            zoom_level=12,
            basemap_source=MACRO_BASEMAP,
        )
        plot_single_map(
            gdf_ozp,
            "Outline Zoning Plan (Kowloon)",
            plot_dir / "Fig2a_OZP_Zoom.png",
            KOWLOON_BOUNDS,
            style_ozp_zoom,
            color_col="DESC_ENG",
            zoom_level=15,
            basemap_source=MICRO_BASEMAP,
        )
    else:
        print("[WARN] OZP source directory not found.")

    print(f"[INFO] All maps finished. Output directory: {plot_dir}")


if __name__ == "__main__":
    run_all_maps()

[INFO] Project base directory: d:\HKBFETD
[INFO] Plot output directory: d:\HKBFETD\plot
[1/5] Processing LandsD...
[INFO] Saved: d:\HKBFETD\plot\Fig2a_LandsD_Full.png
[INFO] Saved: d:\HKBFETD\plot\Fig2a_LandsD_Zoom.png
[2/5] Processing BDBIAR...
[INFO] Saved: d:\HKBFETD\plot\Fig2a_BDBIAR_Full.png
[INFO] Saved: d:\HKBFETD\plot\Fig2a_BDBIAR_Zoom.png
[3/5] Processing OSM...
[INFO] Saved: d:\HKBFETD\plot\Fig2a_OSM_Full.png
[INFO] Saved: d:\HKBFETD\plot\Fig2a_OSM_Zoom.png
[4/5] Processing Overture...
[INFO] Saved: d:\HKBFETD\plot\Fig2a_Overture_Full.png
[INFO] Saved: d:\HKBFETD\plot\Fig2a_Overture_Zoom.png
[5/5] Processing OZP...
[INFO] Loading OZP Esri JSON files...
[INFO] Saved: d:\HKBFETD\plot\Fig2a_OZP_Full.png
[INFO] Saved: d:\HKBFETD\plot\Fig2a_OZP_Zoom.png
[INFO] All maps finished. Output directory: d:\HKBFETD\plot


In [ ]:
# -*- coding: utf-8 -*-

from pathlib import Path

import geopandas as gpd
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.patches import Patch
import contextily as cx


def resolve_project_root() -> Path:
    """
    Resolve the project root when config.py cannot be imported.
    """
    start = Path.cwd()
    candidates = [start, start.parent]

    for candidate in candidates:
        if (candidate / "output").exists() and (candidate / "intermediate_files").exists():
            return candidate

    return start


try:
    from config import (
        BASE_DIR as CONFIG_BASE_DIR,
        AGGREGATED_GDF_PATH,
        OSM_CLEAN_PATH,
    )

    BASE_DIR = Path(CONFIG_BASE_DIR)
    AGGREGATED_PATH = Path(AGGREGATED_GDF_PATH)
    VGI_CLEAN_PATH = Path(OSM_CLEAN_PATH)

except ImportError:
    BASE_DIR = resolve_project_root()
    intermediate_dir = BASE_DIR / "intermediate_files"

    aggregated_candidates = [
        intermediate_dir / "step2_osm_aggregated_buildings.geojson",
        intermediate_dir / "gdf_aggregated.geojson",
    ]
    vgi_clean_candidates = [
        intermediate_dir / "step2_osm_features_clean.geojson",
        intermediate_dir / "gdf_osm_clean.geojson",
    ]

    AGGREGATED_PATH = None
    for candidate in aggregated_candidates:
        if candidate.exists():
            AGGREGATED_PATH = candidate
            break
    if AGGREGATED_PATH is None:
        AGGREGATED_PATH = aggregated_candidates[0]

    VGI_CLEAN_PATH = None
    for candidate in vgi_clean_candidates:
        if candidate.exists():
            VGI_CLEAN_PATH = candidate
            break
    if VGI_CLEAN_PATH is None:
        VGI_CLEAN_PATH = vgi_clean_candidates[0]


PLOT_DIR = BASE_DIR / "plot"
PLOT_DIR.mkdir(parents=True, exist_ok=True)


def set_nature_style():
    """
    Apply a publication-oriented plotting style.
    """
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
        "font.size": 12,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "axes.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    })


COLOR_MATCHED = "#4DBBD5"
COLOR_UNMATCHED = "#E64B35"
COLOR_VGI_POLYGON = "#00A087"
COLOR_VGI_POINT = "#F39B7F"


def make_square_bounds(bounds, scale=0.6):
    """
    Build a square plotting window centered on the target geometry bounds.
    """
    minx, miny, maxx, maxy = bounds
    width = maxx - minx
    height = maxy - miny
    center_x = (minx + maxx) / 2
    center_y = (miny + maxy) / 2
    max_dim = max(width, height)
    buffer = max_dim * scale

    return (
        center_x - buffer,
        center_x + buffer,
        center_y - buffer,
        center_y + buffer,
    )


def plot_figure_x_vgi_integration():
    """
    Plot the 1x3 VGI integration figure with:
    (b) encapsulated VGI polygons,
    (c) encapsulated VGI points,
    (d) richness distribution of VGI integration.
    """
    print("[INFO] Searching for representative VGI polygon and POI integration cases...")

    if not AGGREGATED_PATH.exists() or not VGI_CLEAN_PATH.exists():
        print("[ERROR] Required input files are missing.")
        print(f"[ERROR] Aggregated file: {AGGREGATED_PATH}")
        print(f"[ERROR] Clean VGI file: {VGI_CLEAN_PATH}")
        return

    gdf_agg = gpd.read_file(AGGREGATED_PATH)
    gdf_vgi_all = gpd.read_file(VGI_CLEAN_PATH)

    if gdf_vgi_all.crs != gdf_agg.crs:
        gdf_vgi_all = gdf_vgi_all.to_crs(gdf_agg.crs)

    vgi_counts = gdf_agg.groupby("BUILDINGSTRUCTUREID")["osmid"].count()

    # Select medium-to-large candidate buildings with relatively rich VGI coverage.
    large_buildings = (
        gdf_agg[
            (gdf_agg.geometry.area > 2000) &
            (gdf_agg.geometry.area < 10000)
        ]
        .drop_duplicates("BUILDINGSTRUCTUREID")
        .copy()
    )

    large_buildings["vgi_count"] = (
        large_buildings["BUILDINGSTRUCTUREID"]
        .map(vgi_counts)
        .fillna(0)
    )

    potential_candidates = large_buildings[
        large_buildings["vgi_count"] >= 5
    ].sort_values("vgi_count", ascending=False)

    # Search for the best polygon example.
    target_building_poly = None
    final_vgis_poly = None

    for _, building in potential_candidates.iterrows():
        tolerant_polygon = building.geometry.buffer(2.0)
        contained_vgis = gdf_vgi_all[gdf_vgi_all.geometry.within(tolerant_polygon)]

        polygon_vgis = contained_vgis[
            contained_vgis.geometry.type.isin(["Polygon", "MultiPolygon"])
        ]
        polygon_vgis = polygon_vgis[polygon_vgis.geometry.area < 500]

        if len(polygon_vgis) >= 5:
            target_building_poly = gdf_agg[
                gdf_agg["BUILDINGSTRUCTUREID"] == building["BUILDINGSTRUCTUREID"]
            ].iloc[[0]]
            final_vgis_poly = polygon_vgis
            break

    # Search for the best point-based POI example.
    target_building_poi = None
    final_vgis_poi = None

    for _, building in potential_candidates.iterrows():
        if (
            target_building_poly is not None
            and building["BUILDINGSTRUCTUREID"]
            == target_building_poly.iloc[0]["BUILDINGSTRUCTUREID"]
        ):
            continue

        tolerant_polygon = building.geometry.buffer(2.0)
        contained_vgis = gdf_vgi_all[gdf_vgi_all.geometry.within(tolerant_polygon)]
        point_vgis = contained_vgis[contained_vgis.geometry.type == "Point"]

        if len(point_vgis) >= 8:
            target_building_poi = gdf_agg[
                gdf_agg["BUILDINGSTRUCTUREID"] == building["BUILDINGSTRUCTUREID"]
            ].iloc[[0]]
            final_vgis_poi = point_vgis
            break

    set_nature_style()

    fig = plt.figure(figsize=(17, 3.5))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1.1], wspace=0.1)

    # Panel (b): encapsulated VGI polygons
    ax1 = fig.add_subplot(gs[0])

    if target_building_poly is not None:
        final_vgis_poly.plot(
            ax=ax1,
            facecolor=COLOR_VGI_POLYGON,
            edgecolor="white",
            alpha=0.7,
            linewidth=0.8,
            zorder=2,
        )
        target_building_poly.plot(
            ax=ax1,
            facecolor="none",
            edgecolor=COLOR_UNMATCHED,
            linewidth=3.5,
            zorder=3,
        )

        x_min, x_max, y_min, y_max = make_square_bounds(
            target_building_poly.geometry.total_bounds,
            scale=0.6,
        )
        ax1.set_xlim(x_min, x_max)
        ax1.set_ylim(y_min, y_max)

        try:
            cx.add_basemap(
                ax1,
                crs=target_building_poly.crs.to_string(),
                source=cx.providers.Esri.WorldImagery,
                zoom=18,
            )
        except Exception:
            pass

    ax1.set_xticks([])
    ax1.set_yticks([])
    ax1.set_aspect("equal")
    ax1.set_title("(b) Encapsulated VGI Polygons", loc="left", y=1.02)

    # Panel (c): encapsulated VGI points
    ax2 = fig.add_subplot(gs[1])

    if target_building_poi is not None:
        target_building_poi.plot(
            ax=ax2,
            facecolor="none",
            edgecolor=COLOR_UNMATCHED,
            linewidth=3.5,
            zorder=3,
        )
        final_vgis_poi.plot(
            ax=ax2,
            color=COLOR_VGI_POINT,
            markersize=80,
            edgecolor="white",
            linewidth=1.0,
            zorder=4,
        )

        x_min, x_max, y_min, y_max = make_square_bounds(
            target_building_poi.geometry.total_bounds,
            scale=0.6,
        )
        ax2.set_xlim(x_min, x_max)
        ax2.set_ylim(y_min, y_max)

        try:
            cx.add_basemap(
                ax2,
                crs=target_building_poi.crs.to_string(),
                source=cx.providers.Esri.WorldImagery,
                zoom=18,
            )
        except Exception:
            pass

    ax2.set_xticks([])
    ax2.set_yticks([])
    ax2.set_aspect("equal")
    ax2.set_title("(c) Encapsulated VGI Points (POIs)", loc="left", y=1.02)

    # Panel (d): richness distribution
    ax3 = fig.add_subplot(gs[2])

    valid_vgi_counts = vgi_counts[vgi_counts > 0]
    counts_to_plot = valid_vgi_counts[valid_vgi_counts <= 30]

    sns.histplot(
        x=counts_to_plot,
        discrete=True,
        color=COLOR_MATCHED,
        alpha=0.8,
        ax=ax3,
    )
    ax3.set_yscale("log")
    ax3.set_xlabel("Number of VGI Overlapped per Building", fontweight="bold")
    ax3.set_ylabel("Count of Buildings (Log Scale)", fontweight="bold")
    ax3.set_title("(d) Richness of Crowdsourced VGI Integration", loc="left", y=1.02)
    ax3.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    global_legend = [
        Patch(
            facecolor="none",
            edgecolor=COLOR_UNMATCHED,
            linewidth=2.5,
            label="LandsD Host (Boundary)",
        ),
        Patch(
            facecolor=COLOR_VGI_POLYGON,
            edgecolor="white",
            alpha=0.7,
            label="VGI Polygons",
        ),
        mlines.Line2D(
            [],
            [],
            color="white",
            markerfacecolor=COLOR_VGI_POINT,
            marker="o",
            markersize=11,
            label="VGI Points (POIs)",
        ),
    ]

    fig.legend(
        handles=global_legend,
        loc="lower center",
        bbox_to_anchor=(0.34, -0.03),
        ncol=3,
        frameon=False,
        fontsize=12,
    )

    plt.tight_layout(rect=[0, 0.08, 1, 1])

    out_path = PLOT_DIR / "Fig2_bcd_VGI_Integration_3Cols.png"
    plt.savefig(out_path)
    plt.close(fig)

    print(f"[INFO] Fig. 2(b-d) saved successfully: {out_path}")


if __name__ == "__main__":
    plot_figure_x_vgi_integration()

[INFO] Searching for representative VGI polygon and POI integration cases...


C:\Users\Yizheng\AppData\Local\Temp\ipykernel_231152\350513830.py:351: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.08, 1, 1])


[INFO] Fig. 2(b-d) saved successfully: d:\HKBFETD\plot\Fig2_bcd_VGI_Integration_3Cols.png


In [32]:
# -*- coding: utf-8 -*-
from pathlib import Path
import plotly.graph_objects as go


COLOR_MAP = {
    "Residential": "#4DBBD5",
    "Commercial": "#E64B35",
    "Mixed-use": "#FFC107",
    "Industrial": "#3C5488",
    "Non-assessed": "#B0BCC6",
    "Raw_Data": "#2F4F4F",
    "Rule_Engine": "#008080",
    "LLM_Arb": "#D2691E",
    "Vision_AI": "#6A5ACD",
    "ML_Inf": "#2E8B57",
}


def hex_to_rgba(hex_color, alpha=0.4):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {alpha})"


def resolve_project_root() -> Path:
    start = Path.cwd()
    candidates = [start, start.parent]
    for candidate in candidates:
        if (candidate / "output").exists() and (candidate / "intermediate_files").exists():
            return candidate
    return start


try:
    from config import BASE_DIR as CONFIG_BASE_DIR
    BASE_DIR = Path(CONFIG_BASE_DIR)
except ImportError:
    BASE_DIR = resolve_project_root()

PLOT_DIR = BASE_DIR / "plot"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

HTML_OUT = PLOT_DIR / "Fig4_Sankey.html"
PNG_OUT = PLOT_DIR / "Fig4_Sankey.png"


node_labels = [
    "Raw Footprints<br>(341,153)",               # 0
    "Knowledge-driven<br>Rule Engine (341,153)", # 1
    "Rule-based Baseline<br>Retained (146,774)", # 2
    "LLM Semantic<br>Arbitration (12,453)",      # 3
    "VLM Calibration<br>(1,510)",                # 4
    "Probabilistic ML<br>Inference (180,416)",   # 5
    "Residential<br>(251,901)",                  # 6
    "Commercial<br>(49,129)",                    # 7
    "Mixed-use<br>(31,395)",                     # 8
    "Industrial<br>(3,355)",                     # 9
    "Non-assessed<br>(5,373)",                   # 10
]

node_colors = [
    COLOR_MAP["Raw_Data"],      # 0
    COLOR_MAP["Rule_Engine"],   # 1
    COLOR_MAP["Rule_Engine"],   # 2
    COLOR_MAP["LLM_Arb"],       # 3
    COLOR_MAP["Vision_AI"],     # 4
    COLOR_MAP["ML_Inf"],        # 5
    COLOR_MAP["Residential"],   # 6
    COLOR_MAP["Commercial"],    # 7
    COLOR_MAP["Mixed-use"],     # 8
    COLOR_MAP["Industrial"],    # 9
    COLOR_MAP["Non-assessed"],  # 10
]

sources, targets, values, link_colors = [], [], [], []


def add_link(src, tgt, val, color_hex, alpha=0.35):
    if val <= 0:
        return
    sources.append(src)
    targets.append(tgt)
    values.append(val)
    link_colors.append(hex_to_rgba(color_hex, alpha))


# Backbone
add_link(0, 1, 341153, COLOR_MAP["Rule_Engine"], 0.45)

add_link(1, 2, 146774, COLOR_MAP["Rule_Engine"], 0.42)
add_link(1, 3, 12453,  COLOR_MAP["LLM_Arb"], 0.42)
add_link(1, 4, 1510,   COLOR_MAP["Vision_AI"], 0.35)
add_link(1, 5, 180416, COLOR_MAP["ML_Inf"], 0.45)

# Rule-based baseline retained -> final classes
add_link(2, 6, 122455, COLOR_MAP["Residential"])
add_link(2, 7, 13919,  COLOR_MAP["Commercial"])
add_link(2, 8, 6043,   COLOR_MAP["Mixed-use"])
add_link(2, 9, 804,    COLOR_MAP["Industrial"])
add_link(2, 10, 3553,  COLOR_MAP["Non-assessed"])

# LLM -> final classes
add_link(3, 6, 2528, COLOR_MAP["Residential"])
add_link(3, 7, 8313, COLOR_MAP["Commercial"])
add_link(3, 8, 17,   COLOR_MAP["Mixed-use"])
add_link(3, 9, 323,  COLOR_MAP["Industrial"])
add_link(3, 10, 1272, COLOR_MAP["Non-assessed"])

# VLM -> final classes
add_link(4, 6, 889, COLOR_MAP["Residential"], 0.30)
add_link(4, 7, 514, COLOR_MAP["Commercial"], 0.30)
add_link(4, 8, 60,  COLOR_MAP["Mixed-use"], 0.30)
add_link(4, 9, 43,  COLOR_MAP["Industrial"], 0.30)
add_link(4, 10, 4,  COLOR_MAP["Non-assessed"], 0.30)

# ML -> final classes
add_link(5, 6, 126029, COLOR_MAP["Residential"])
add_link(5, 7, 26383,  COLOR_MAP["Commercial"])
add_link(5, 8, 25275,  COLOR_MAP["Mixed-use"])
add_link(5, 9, 2185,   COLOR_MAP["Industrial"])
add_link(5, 10, 544,   COLOR_MAP["Non-assessed"])


fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=32,
        thickness=62,
        line=dict(color="black", width=0.8),
        label=node_labels,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    font=dict(size=20, family="Times New Roman", color="black"),
    width=1500,
    height=850,
    margin=dict(l=40, r=40, t=30, b=30),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

html_str = fig.to_html(include_plotlyjs="cdn", full_html=True)

css_injection = """
<style>
    text, .sankey-node text, .sankey-link text {
        text-shadow: none !important;
        font-weight: 900 !important;
        fill: #000000 !important;
    }
</style>
"""
html_str = html_str.replace("</head>", css_injection + "</head>")

with open(HTML_OUT, "w", encoding="utf-8") as f:
    f.write(html_str)

print(f"Saved HTML to: {HTML_OUT}")

try:
    fig.write_image(str(PNG_OUT), scale=2)
    print(f"Saved PNG to: {PNG_OUT}")
except Exception:
    print("PNG export skipped. Install kaleido if needed:")
    print("pip install kaleido")

Saved HTML to: d:\HKBFETD\plot\Fig4_Sankey.html


Wait expired, Browser is being closed by watchdog.


PNG export skipped. Install kaleido if needed:
pip install kaleido


In [18]:
# -*- coding: utf-8 -*-

import os
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import contextily as cx
from shapely.geometry import Point


def resolve_project_root() -> Path:
    """
    Resolve the project root when config.py cannot be imported.
    """
    start = Path.cwd()
    candidates = [start, start.parent]

    for candidate in candidates:
        if (candidate / "output").exists() and (candidate / "intermediate_files").exists():
            return candidate

    return start


def patch_proj_environment():
    """
    Patch PROJ environment variables for Windows conda environments.
    """
    conda_prefix = sys.prefix
    proj_lib_path = os.path.join(conda_prefix, "Library", "share", "proj")

    if os.path.exists(proj_lib_path):
        os.environ["PROJ_LIB"] = proj_lib_path
        os.environ["PROJ_DATA"] = proj_lib_path


patch_proj_environment()

try:
    from config import BASE_DIR as CONFIG_BASE_DIR, OUTPUT_DIR as CONFIG_OUTPUT_DIR

    BASE_DIR = Path(CONFIG_BASE_DIR)
    OUTPUT_DIR = Path(CONFIG_OUTPUT_DIR)

except ImportError:
    BASE_DIR = resolve_project_root()
    OUTPUT_DIR = BASE_DIR / "output"

PLOT_DIR = BASE_DIR / "plot"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

PUBLIC_GEOJSON = OUTPUT_DIR / "HK_UBEM_Buildings_Public_v1.geojson"


def set_nature_style():
    """
    Apply a publication-oriented plotting style.
    """
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
        "font.size": 12,
        "axes.titleweight": "bold",
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    })


COLOR_MAP = {
    "Residential": "#4DBBD5",
    "Commercial": "#E64B35",
    "Mixed-use": "#FFC107",
    "Industrial": "#3C5488",
    "Non-assessed": "#B0BCC6",
}


def create_bbox(center_lon, center_lat, radius_y_m, aspect_ratio, crs_target):
    """
    Create a bounding box using a vertical radius and a target aspect ratio.
    """
    center_point = (
        gpd.GeoSeries([Point(center_lon, center_lat)], crs="EPSG:4326")
        .to_crs(crs_target)
        .iloc[0]
    )

    radius_x_m = radius_y_m * aspect_ratio

    minx = center_point.x - radius_x_m
    maxx = center_point.x + radius_x_m
    miny = center_point.y - radius_y_m
    maxy = center_point.y + radius_y_m

    return minx, miny, maxx, maxy


def plot_regional_map(ax, gdf, bbox, title, edge_width=0.0):
    """
    Render building polygons within a specified bounding box.
    """
    minx, miny, maxx, maxy = bbox
    subset = gdf.cx[minx:maxx, miny:maxy]

    for class_key, class_color in COLOR_MAP.items():
        class_subset = subset[
            subset["UBEM_Main_Class_主类别"].astype(str).str.contains(class_key, na=False)
        ]
        if not class_subset.empty:
            if edge_width > 0:
                class_subset.plot(
                    ax=ax,
                    color=class_color,
                    edgecolor="white",
                    linewidth=edge_width,
                )
            else:
                class_subset.plot(
                    ax=ax,
                    color=class_color,
                    edgecolor="none",
                )

    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    try:
        cx.add_basemap(
            ax,
            crs=gdf.crs.to_string(),
            source=cx.providers.Esri.WorldImagery,
            alpha=0.7,
            attribution=False,
        )
    except Exception:
        print(f"[WARN] Failed to load basemap for: {title}")

    ax.set_title(title, loc="left", pad=8, fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])


def draw_reference_box(ax, bbox_to_draw, color="white", linewidth=2.5, linestyle="--"):
    """
    Draw a reference rectangle on the map.
    """
    minx, miny, maxx, maxy = bbox_to_draw
    rect = plt.Rectangle(
        (minx, miny),
        maxx - minx,
        maxy - miny,
        fill=False,
        color=color,
        linewidth=linewidth,
        linestyle=linestyle,
        zorder=10,
    )
    ax.add_patch(rect)


def generate_citywide_showcase():
    """
    Generate the multi-scale spatial mapping figure for the public dataset.
    """
    print("=" * 70)
    print("[INFO] Generating multi-scale spatial mapping figure...")
    print("=" * 70)

    if not PUBLIC_GEOJSON.exists():
        print(f"[ERROR] Final public GeoJSON not found: {PUBLIC_GEOJSON}")
        return

    gdf = gpd.read_file(PUBLIC_GEOJSON)
    set_nature_style()

    fig = plt.figure(figsize=(16, 12))
    gs = gridspec.GridSpec(6, 2, width_ratios=[1.3, 1], wspace=0.01, hspace=0.2)

    ax_hk = fig.add_subplot(gs[0:3, 0])
    ax_kln = fig.add_subplot(gs[3:6, 0])
    ax_mk = fig.add_subplot(gs[0:2, 1])
    ax_tst = fig.add_subplot(gs[2:4, 1])
    ax_hmt = fig.add_subplot(gs[4:6, 1])

    # Compute adaptive bounding boxes
    hk_bounds = gdf.total_bounds
    width_hk = hk_bounds[2] - hk_bounds[0]
    height_hk = hk_bounds[3] - hk_bounds[1]
    bbox_hk = (
        hk_bounds[0] - width_hk * 0.05,
        hk_bounds[1] - height_hk * 0.05,
        hk_bounds[2] + width_hk * 0.05,
        hk_bounds[3] + height_hk * 0.05,
    )

    left_col_aspect = (bbox_hk[2] - bbox_hk[0]) / (bbox_hk[3] - bbox_hk[1])

    bbox_kln = create_bbox(
        center_lon=114.18,
        center_lat=22.32,
        radius_y_m=3500,
        aspect_ratio=left_col_aspect,
        crs_target=gdf.crs,
    )

    bbox_mk = create_bbox(
        center_lon=114.170,
        center_lat=22.319,
        radius_y_m=700,
        aspect_ratio=1.0,
        crs_target=gdf.crs,
    )
    bbox_tst = create_bbox(
        center_lon=114.172,
        center_lat=22.298,
        radius_y_m=700,
        aspect_ratio=1.0,
        crs_target=gdf.crs,
    )
    bbox_hmt = create_bbox(
        center_lon=114.185,
        center_lat=22.315,
        radius_y_m=700,
        aspect_ratio=1.0,
        crs_target=gdf.crs,
    )

    # Render maps
    plot_regional_map(
        ax_hk,
        gdf,
        bbox_hk,
        "(a) City-wide Territorial Coverage",
        edge_width=0.0,
    )
    draw_reference_box(ax_hk, bbox_kln, color="white", linewidth=2.5, linestyle="-")

    plot_regional_map(
        ax_kln,
        gdf,
        bbox_kln,
        "(b) Core Urban District (Kowloon Peninsula)",
        edge_width=0.0,
    )
    draw_reference_box(ax_kln, bbox_mk, color="white", linewidth=2)
    draw_reference_box(ax_kln, bbox_tst, color="white", linewidth=2)
    draw_reference_box(ax_kln, bbox_hmt, color="white", linewidth=2)

    plot_regional_map(
        ax_mk,
        gdf,
        bbox_mk,
        "(c) Mong Kok (Dense Mixed-use Hub)",
        edge_width=0.3,
    )
    plot_regional_map(
        ax_tst,
        gdf,
        bbox_tst,
        "(d) Tsim Sha Tsui (Commercial Core)",
        edge_width=0.3,
    )
    plot_regional_map(
        ax_hmt,
        gdf,
        bbox_hmt,
        "(e) Ho Man Tin (Residential Zone)",
        edge_width=0.3,
    )

    legend_elements = [
        Patch(facecolor=COLOR_MAP["Residential"], label="Residential"),
        Patch(facecolor=COLOR_MAP["Commercial"], label="Commercial"),
        Patch(facecolor=COLOR_MAP["Mixed-use"], label="Mixed-use"),
        Patch(facecolor=COLOR_MAP["Industrial"], label="Industrial"),
        Patch(facecolor=COLOR_MAP["Non-assessed"], label="Non-assessed"),
    ]

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=5,
        frameon=False,
        fontsize=14,
        handlelength=1.5,
        handleheight=1.5,
    )

    plt.tight_layout(rect=[0, 0.06, 1, 1])

    out_path = PLOT_DIR / "Fig5_MultiScale_Spatial_Mapping.png"
    plt.savefig(out_path, dpi=300)
    plt.close(fig)

    print(f"[INFO] Fig. 5 saved successfully: {out_path}")


if __name__ == "__main__":
    generate_citywide_showcase()

[INFO] Generating multi-scale spatial mapping figure...


C:\Users\Yizheng\AppData\Local\Temp\ipykernel_231152\1898924204.py:295: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.06, 1, 1])


[INFO] Fig. 5 saved successfully: d:\HKBFETD\plot\Fig5_MultiScale_Spatial_Mapping.png


In [ ]:
# -*- coding: utf-8 -*-

import os
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns


def resolve_project_root() -> Path:
    """
    Resolve the project root when config.py cannot be imported.
    """
    start = Path.cwd()
    candidates = [start, start.parent]

    for candidate in candidates:
        if (candidate / "output").exists() and (candidate / "intermediate_files").exists():
            return candidate

    return start


try:
    from config import BASE_DIR as CONFIG_BASE_DIR, OFFICIAL_LIB_BASE_PATH

    BASE_DIR = Path(CONFIG_BASE_DIR)
    OFFICIAL_BASE_PATH = Path(OFFICIAL_LIB_BASE_PATH)

except ImportError:
    BASE_DIR = resolve_project_root()
    intermediate_dir = BASE_DIR / "intermediate_files"

    fallback_candidates = [
        intermediate_dir / "step1_official_building_library_base.geojson",
        intermediate_dir / "gdf_official_library_base.geojson",
    ]

    OFFICIAL_BASE_PATH = None
    for candidate in fallback_candidates:
        if candidate.exists():
            OFFICIAL_BASE_PATH = candidate
            break

    if OFFICIAL_BASE_PATH is None:
        OFFICIAL_BASE_PATH = fallback_candidates[0]

PLOT_DIR = BASE_DIR / "plot"
PLOT_DIR.mkdir(parents=True, exist_ok=True)


def set_nature_style():
    """
    Apply a publication-oriented plotting style.
    """
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
        "font.size": 12,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "axes.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    })


COLOR_MATCHED = "#4DBBD5"
COLOR_UNMATCHED = "#E64B35"
COLOR_VILLAGE_SHADOW = "#FFB74D"
COLOR_VILLAGE_LINE = "#FF8C00"


def plot_figure_y_baseline_coverage():
    """
    Plot the baseline coverage and long-tail morphology figure.
    """
    print("[INFO] Loading baseline data for Fig. 6...")

    if not OFFICIAL_BASE_PATH.exists():
        print(f"[ERROR] Official base library not found: {OFFICIAL_BASE_PATH}")
        return

    gdf = gpd.read_file(OFFICIAL_BASE_PATH)

    # Use official GFA when available.
    if "GFA_DOMESTIC_SUM" in gdf.columns and "GFA_NONDOMESTIC_SUM" in gdf.columns:
        official_gfa = gdf["GFA_DOMESTIC_SUM"].fillna(0) + gdf["GFA_NONDOMESTIC_SUM"].fillna(0)
    else:
        official_gfa = pd.Series(0, index=gdf.index)

    # Build a fallback proxy GFA for records without official GFA.
    if "Floors_Count" in gdf.columns:
        proxy_gfa = gdf.geometry.area * gdf["Floors_Count"].fillna(1)
    else:
        proxy_gfa = gdf.geometry.area * 3

    # Use official GFA where available; otherwise use the proxy estimate.
    gdf["Total_GFA"] = np.where(official_gfa > 0, official_gfa, proxy_gfa)

    matched_mask = (
        gdf["BDBIAR_OBJECTID"].notna()
        & (gdf["BDBIAR_OBJECTID"].astype(str) != "")
        & (gdf["BDBIAR_OBJECTID"].astype(str) != "nan")
    )

    df_matched = gdf[matched_mask]
    df_unmatched = gdf[~matched_mask]

    count_matched = len(df_matched)
    count_unmatched = len(df_unmatched)
    gfa_matched = df_matched["Total_GFA"].sum()
    gfa_unmatched = df_unmatched["Total_GFA"].sum()

    set_nature_style()

    fig = plt.figure(figsize=(15, 6))
    gs = gridspec.GridSpec(
        2,
        3,
        width_ratios=[1.1, 1.1, 0.8],
        hspace=0.15,
        wspace=0.1,
    )

    # Panel A: footprint-area long-tail distribution
    ax1 = fig.add_subplot(gs[:, 0:2])

    area_matched = df_matched.geometry.area.clip(upper=2000)
    area_unmatched = df_unmatched.geometry.area.clip(upper=2000)

    sns.histplot(
        area_unmatched,
        bins=60,
        color=COLOR_UNMATCHED,
        element="poly",
        fill=True,
        alpha=0.5,
        label="Unmatched (LandsD Only)",
        ax=ax1,
        log_scale=(False, True),
    )
    sns.histplot(
        area_matched,
        bins=60,
        color=COLOR_MATCHED,
        element="poly",
        fill=True,
        alpha=0.5,
        label="Matched (LandsD + BD)",
        ax=ax1,
        log_scale=(False, True),
    )

    ax1.axvspan(30, 80, color=COLOR_VILLAGE_SHADOW, alpha=0.3, zorder=0)
    ax1.axvline(x=65, color=COLOR_VILLAGE_LINE, linestyle="--", linewidth=1.5)

    ymin = ax1.get_ylim()[0]
    ax1.text(
        80,
        ymin * 3,
        "Village House Trap\n(~65 m²)",
        color="black",
        fontsize=12,
        va="bottom",
    )

    ax1.set_xlabel("Building Footprint Area (m²)", fontweight="bold")
    ax1.set_ylabel("Count of Buildings (Log Scale)", fontweight="bold")
    ax1.set_title("(a) Morphological Dichotomy and Long-tail Effect", loc="left", y=1.02)
    ax1.legend(frameon=False, loc="upper right")

    # Panel B: building-count donut chart
    colors = [COLOR_MATCHED, COLOR_UNMATCHED]
    explode = (0.05, 0)

    ax2 = fig.add_subplot(gs[0, 2])
    ax2.pie(
        [count_matched, count_unmatched],
        explode=explode,
        labels=None,
        colors=colors,
        autopct="%1.2f%%",
        radius=1.2,
        startangle=90,
        textprops=dict(color="k", weight="bold", size=11),
    )
    centre_circle1 = plt.Circle((0, 0), 0.60, fc="white")
    ax2.add_artist(centre_circle1)
    ax2.text(0, 0, "Building\nCount", ha="center", va="center", fontweight="bold", fontsize=12)
    ax2.set_title("(b) Macro-scale Coverage", loc="left", y=1.05)

    # Panel C: total-GFA donut chart
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.pie(
        [gfa_matched, gfa_unmatched],
        explode=explode,
        labels=None,
        colors=colors,
        autopct="%1.2f%%",
        radius=1.2,
        startangle=90,
        textprops=dict(color="k", weight="bold", size=11),
    )
    centre_circle2 = plt.Circle((0, 0), 0.60, fc="white")
    ax3.add_artist(centre_circle2)
    ax3.text(0, 0, "Total\nGFA", ha="center", va="center", fontweight="bold", fontsize=12)

    plt.tight_layout(rect=[0, 0, 0.92, 1], w_pad=1.0)

    out_path = PLOT_DIR / "Fig6_Morphological_Distribution.png"
    plt.savefig(out_path)
    plt.close(fig)

    print(f"[INFO] Fig. 6 saved successfully: {out_path}")


if __name__ == "__main__":
    plot_figure_y_baseline_coverage()

[INFO] Loading baseline data for Fig. 6...


C:\Users\Yizheng\AppData\Local\Temp\ipykernel_231152\3679006481.py:221: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.92, 1], w_pad=1.0)


[INFO] Fig. 6 saved successfully: d:\HKBFETD\plot\Fig6_Morphological_Distribution.png


In [27]:
# -*- coding: utf-8 -*-
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np


def resolve_project_root() -> Path:
    """
    Resolve the project root when config.py cannot be imported.
    """
    start = Path.cwd()
    candidates = [start, start.parent]

    for candidate in candidates:
        if (candidate / "output").exists() and (candidate / "intermediate_files").exists():
            return candidate

    return start


try:
    from config import BASE_DIR as CONFIG_BASE_DIR
    BASE_DIR = Path(CONFIG_BASE_DIR)
except ImportError:
    BASE_DIR = resolve_project_root()

PLOT_DIR = BASE_DIR / "plot"
PLOT_DIR.mkdir(parents=True, exist_ok=True)


def set_nature_style():
    """
    Apply a publication-oriented plotting style.
    Font size is intentionally fixed at 13.
    """
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif", "serif"],
        "font.size": 13,
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "legend.fontsize": 13,
        "axes.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
    })


COLOR_RULE = "#B0BEC5"
COLOR_AI_RAW = "#4DBBD5"
COLOR_APPLIED = "#00A087"

COLOR_RES = "#E64B35"
COLOR_COM = "#4DBBD5"
COLOR_IND = "#3C5488"


def plot_figure_validation():
    """
    Plot the technical validation summary figure.
    """
    set_nature_style()

    fig = plt.figure(figsize=(16, 13))

    # The second row is half the height of the first row.
    gs = gridspec.GridSpec(
        2,
        2,
        width_ratios=[1.2, 1],
        height_ratios=[1, 0.5],
        wspace=0.2,
        hspace=0.3,
    )

    # ========================================================
    # Panel (a): arbitration performance gain
    # ========================================================
    ax1 = fig.add_subplot(gs[0, 0])

    acc_sem = [58.0, 79.0, 81.0]
    acc_geo = [57.0, 70.0]

    x_sem = [1, 2, 3]
    x_geo = [5, 6]

    b1 = ax1.bar(
        x_sem[0], acc_sem[0],
        color=COLOR_RULE, width=0.7, edgecolor="black", linewidth=0.8,
        label="Rule Engine Baseline"
    )
    b2 = ax1.bar(
        x_sem[1], acc_sem[1],
        color=COLOR_AI_RAW, width=0.7, edgecolor="black", linewidth=0.8,
        label="Raw AI Inference"
    )
    b3 = ax1.bar(
        x_sem[2], acc_sem[2],
        color=COLOR_APPLIED, width=0.7, edgecolor="black", linewidth=0.8,
        label="Final Hybrid Applied"
    )
    b4 = ax1.bar(
        x_geo[0], acc_geo[0],
        color=COLOR_AI_RAW, width=0.7, edgecolor="black", linewidth=0.8
    )
    b5 = ax1.bar(
        x_geo[1], acc_geo[1],
        color=COLOR_APPLIED, width=0.7, edgecolor="black", linewidth=0.8
    )

    for bars in [b1, b2, b3, b4, b5]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(
                bar.get_x() + bar.get_width() / 2.0,
                height + 1.5,
                f"{height:.1f}%",
                ha="center",
                va="bottom",
                fontsize=13,
                fontweight="bold",
            )

    ax1.set_ylim(0, 105)
    ax1.set_xticks([2, 5.5])
    ax1.set_xticklabels(
        ["Semantic Arbitration\n(Text-based)", "Geometric Calibration\n(Vision-based)"],
        fontweight="bold",
    )
    ax1.set_ylabel("Ground-Truth Accuracy (%)", fontweight="bold")
    ax1.set_title("(a) Multi-stage System Performance Gain", loc="left")
    ax1.legend(loc="upper right", frameon=False)

    # The gs[0, 1] slot remains available for additional map panels if needed.

    # ========================================================
    # Panel (c): ML classification performance
    # ========================================================
    ax2 = fig.add_subplot(gs[1, 0])

    classes = ["Industrial", "Commercial", "Residential"]
    f1_scores = [71.6, 76.7, 92.2]
    colors = [COLOR_IND, COLOR_COM, COLOR_RES]

    y_pos = np.arange(len(classes))
    bars_f1 = ax2.barh(
        y_pos,
        f1_scores,
        color=colors,
        height=0.6,
        edgecolor="black",
        linewidth=0.8,
    )

    for bar in bars_f1:
        width = bar.get_width()
        ax2.text(
            width - 2,
            bar.get_y() + bar.get_height() / 2.0,
            f"{width:.1f}%",
            ha="right",
            va="center",
            color="white",
            fontsize=13,
            fontweight="bold",
        )

    macro_f1 = 80.2
    ax2.axvline(x=macro_f1, color="gray", linestyle="--", linewidth=1.2, alpha=0.7)
    ax2.text(
        macro_f1 - 0.1,
        2.54,
        f"Macro-F1: {macro_f1:.1f}%",
        color="black",
        fontsize=13,
        fontweight="bold",
    )

    ax2.set_xlim(0, 100)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(classes, fontweight="bold", rotation=90, va="center")
    ax2.set_xlabel("Main-class F1-Score (%)", fontweight="bold")
    ax2.set_title("(c) ML Classification Performance", loc="left")

    # ========================================================
    # Panel (d): subclass proportion prediction
    # ========================================================
    ax3 = fig.add_subplot(gs[1, 1])

    mse_values = [0.028, 0.042, 0.005]

    bars_mse = ax3.barh(
        y_pos,
        mse_values,
        color=colors,
        height=0.6,
        edgecolor="black",
        linewidth=0.8,
        alpha=0.8,
    )

    for bar in bars_mse:
        width = bar.get_width()
        ax3.text(
            width + 0.001,
            bar.get_y() + bar.get_height() / 2.0,
            f"{width:.3f}",
            ha="left",
            va="center",
            color="black",
            fontsize=13,
            fontweight="bold",
        )

    ax3.set_yticks(y_pos)
    ax3.set_yticklabels(classes, fontweight="bold", rotation=90, va="center")
    ax3.set_xlabel("Subclass Regression MSE", fontweight="bold")
    ax3.set_title("(d) Subclass Proportion Prediction", loc="left")
    ax3.set_xlim(0, 0.05)

    plt.tight_layout()

    out_path = PLOT_DIR / "Fig7_Technical_Validation.png"
    plt.savefig(out_path)
    plt.close(fig)

    print(f"[INFO] Fig. 7 saved successfully: {out_path}")


if __name__ == "__main__":
    plot_figure_validation()

C:\Users\Yizheng\AppData\Local\Temp\ipykernel_231152\4198460298.py:229: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


[INFO] Fig. 7 saved successfully: d:\HKBFETD\plot\Fig7_Technical_Validation.png
